# Terrain Dataset & DataLoader

This notebook builds the data pipeline that feeds the preprocessed terrain
patches into the diffusion model. It defines:

1. **`TerrainDataset`** — a PyTorch `Dataset` that loads each 256×256 elevation
   patch (`.npy`) and returns it as a single-channel tensor.
2. **`DataLoader`** — groups patches into shuffled mini-batches for training.

**Input:** 442 normalized patches in `../outputs/processed/patches`.

## 1. Environment check

Confirm this notebook runs in the `diffusion` environment (PyTorch + CUDA).

In [1]:
import sys
print(sys.executable)  # should contain "envs\\diffusion"

c:\Users\ASUS\miniconda3\envs\diffusion\python.exe


## 2. The `TerrainDataset` class

A PyTorch `Dataset` must implement three methods:

| Method | Purpose |
|--------|---------|
| `__init__`    | setup: list all patch files |
| `__len__`     | return the number of patches |
| `__getitem__` | load and return patch *i* as a tensor |

Each patch is a `(256, 256)` array. The model expects a **channel dimension**,
so we reshape it to `(1, 256, 256)` (1 channel = elevation).

In [2]:
import glob
import numpy as np
import torch
from torch.utils.data import Dataset


class TerrainDataset(Dataset):
    """Loads preprocessed 256x256 terrain elevation patches as single-channel tensors."""

    def __init__(self, patch_dir):
        # Sort file paths so patch order is reproducible across runs
        self.files = sorted(glob.glob(patch_dir + "/*.npy"))

    def __len__(self):
        # Number of samples, so the DataLoader knows how far to iterate
        return len(self.files)

    def __getitem__(self, idx):
        patch = np.load(self.files[idx])   # load patch idx -> NumPy array (256, 256)
        tensor = torch.from_numpy(patch)   # NumPy -> PyTorch tensor
        tensor = tensor.unsqueeze(0)       # add channel dim: model expects (C, H, W)
        return tensor

## 3. Verify a single sample

Check that one patch returns the expected type, shape, dtype, and value range
(patches were normalized to roughly `[-1, 1]` during preprocessing).

In [3]:
ds = TerrainDataset("../outputs/processed/patches")

print("Number of patches:", len(ds))     # calls __len__

sample = ds[0]                            # calls __getitem__ automatically
print("Type :", type(sample))             # torch.Tensor
print("Shape:", sample.shape)             # torch.Size([1, 256, 256])
print("Dtype:", sample.dtype)             # torch.float32
print("Range:", sample.min().item(), "->", sample.max().item())  # within ~[-1, 1]

Number of patches: 442
Type : <class 'torch.Tensor'>
Shape: torch.Size([1, 256, 256])
Dtype: torch.float32
Range: -0.10039997100830078 -> 0.4004000425338745


## 4. The `DataLoader`

The `DataLoader` wraps the dataset and serves it in mini-batches, reshuffled
every epoch. We use a small `batch_size` of 8 to fit the 8 GB GPU.

In [4]:
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=8, shuffle=True)   # reuses ds from the cell above

# Pull a single batch to confirm its shape
batch = next(iter(loader))
print("Batch shape:", batch.shape)   # torch.Size([8, 1, 256, 256])

Batch shape: torch.Size([8, 1, 256, 256])
